# Dinámica estacional de los humedales de El Yali

Análisis exploratorio de indicadores ambientales para la Albufera, Laguna Colejuda y Laguna Matanza durante 2019–2025. El cuaderno utiliza tablas analíticas derivadas de composiciones estacionales previamente validadas y se concentra en el control de calidad, la comparación temporal y la comunicación de resultados.

## Alcance

La superficie estimada representa píxeles con respuesta espectral de agua abierta dentro de sectores analíticos. No corresponde a una delimitación legal, una medición oficial ni una estimación completa de ambientes húmedos cubiertos por vegetación. Las relaciones con precipitación son exploratorias y no implican causalidad.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

In [ ]:
# El cuaderno funciona desde la raíz del repositorio o desde notebooks/.
candidatos = [Path.cwd(), Path.cwd().parent]
raiz = next(
    ruta for ruta in candidatos
    if (ruta / "data" / "results" / "sector_analysis").exists()
)

carpeta_datos = raiz / "data" / "results" / "sector_analysis"

serie = pd.read_csv(carpeta_datos / "serie_integrada_por_sector.csv")
sensibilidad = pd.read_csv(carpeta_datos / "sensibilidad_mndwi_por_sector.csv")
superficies = pd.read_csv(carpeta_datos / "superficies_sectores_analiticos.csv")

print(f"Registros cargados: {len(serie)}")
serie.head()

## Control de calidad

Se espera una observación por combinación de zona, año y temporada: tres unidades, siete años y dos temporadas, equivalentes a 42 registros.

In [ ]:
claves = ["zona", "anio", "temporada"]
columnas_analiticas = [
    "superficie_agua_ha",
    "ndvi_medio",
    "precipitacion_mm",
]

control_calidad = pd.Series({
    "registros": len(serie),
    "duplicados": int(serie.duplicated(claves).sum()),
    "valores_ausentes": int(serie[columnas_analiticas].isna().sum().sum()),
    "zonas": serie["zona"].nunique(),
    "anios": serie["anio"].nunique(),
    "temporadas": serie["temporada"].nunique(),
})

assert control_calidad["registros"] == 42
assert control_calidad["duplicados"] == 0
assert control_calidad["valores_ausentes"] == 0
assert serie["ndvi_medio"].between(-1, 1).all()
assert (serie["superficie_agua_ha"] >= 0).all()

control_calidad.to_frame("resultado")

## Indicadores comparables

La superficie de agua se normaliza por el tamaño de cada sector para evitar que la comparación dependa solamente de su extensión.

In [ ]:
datos = serie.merge(
    superficies,
    on="zona",
    how="left",
    validate="many_to_one",
    suffixes=("", "_referencia"),
)

if "superficie_sector_ha_referencia" in datos.columns:
    datos["superficie_sector_ha"] = datos["superficie_sector_ha"].fillna(
        datos["superficie_sector_ha_referencia"]
    )
    datos = datos.drop(columns="superficie_sector_ha_referencia")

datos["agua_abierta_pct"] = (
    datos["superficie_agua_ha"] / datos["superficie_sector_ha"] * 100
)

resumen = (
    datos.groupby(["zona", "temporada"], as_index=False)
    .agg(
        agua_promedio_ha=("superficie_agua_ha", "mean"),
        agua_maxima_ha=("superficie_agua_ha", "max"),
        cobertura_agua_promedio_pct=("agua_abierta_pct", "mean"),
        ndvi_promedio=("ndvi_medio", "mean"),
    )
)

resumen.round(2)

In [ ]:
orden_zonas = ["Albufera", "Laguna Colejuda", "Laguna Matanza"]
colores = {"invierno": "#3B6FB6", "verano": "#E07B39"}

fig, ejes = plt.subplots(1, 3, figsize=(16, 4.6), sharex=True)

for eje, zona in zip(ejes, orden_zonas):
    subconjunto = datos.loc[datos["zona"] == zona]
    sns.lineplot(
        data=subconjunto,
        x="anio",
        y="superficie_agua_ha",
        hue="temporada",
        style="temporada",
        markers=True,
        dashes=False,
        palette=colores,
        ax=eje,
    )
    eje.set(title=zona, xlabel="Año", ylabel="Agua abierta estimada (ha)")
    if zona != "Albufera":
        eje.get_legend().remove()
    else:
        eje.legend(title="Temporada")

fig.suptitle("Variación estacional del agua abierta en El Yali", y=1.02)
plt.tight_layout()
plt.show()

La Albufera presenta agua abierta durante todas las temporadas analizadas. Colejuda muestra una contracción estival marcada, mientras Matanza concentra sus principales pulsos en los inviernos de 2020 y 2024.

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(16, 4.6), sharex=True, sharey=True)

for eje, zona in zip(ejes, orden_zonas):
    subconjunto = datos.loc[datos["zona"] == zona]
    sns.lineplot(
        data=subconjunto,
        x="anio",
        y="ndvi_medio",
        hue="temporada",
        style="temporada",
        markers=True,
        dashes=False,
        palette=colores,
        ax=eje,
    )
    eje.set(title=zona, xlabel="Año", ylabel="NDVI medio")
    if zona != "Albufera":
        eje.get_legend().remove()
    else:
        eje.legend(title="Temporada")

fig.suptitle("Condición media de la vegetación en El Yali", y=1.02)
plt.tight_layout()
plt.show()

El NDVI más alto de las lagunas interiores indica que la ausencia de píxeles clasificados como agua abierta no equivale necesariamente a la ausencia de condiciones húmedas. La vegetación emergente, el agua somera y los sedimentos modifican la respuesta espectral.

## Relación exploratoria con precipitación

Las correlaciones se calculan por zona y temporada. Cada coeficiente se basa en siete observaciones, por lo que se utiliza como descripción del periodo y no como evidencia causal.

In [ ]:
filas_correlacion = []

for (zona, temporada), grupo in datos.groupby(["zona", "temporada"]):
    filas_correlacion.append({
        "zona": zona,
        "temporada": temporada,
        "observaciones": len(grupo),
        "pearson_agua_precipitacion": grupo["superficie_agua_ha"].corr(
            grupo["precipitacion_mm"], method="pearson"
        ),
        "spearman_agua_precipitacion": grupo["superficie_agua_ha"].corr(
            grupo["precipitacion_mm"], method="spearman"
        ),
    })

correlaciones = pd.DataFrame(filas_correlacion)

correlaciones.round(3)

In [ ]:
invierno = datos.loc[datos["temporada"] == "invierno"]
fig, ejes = plt.subplots(1, 3, figsize=(16, 4.6), sharex=True)

for eje, zona in zip(ejes, orden_zonas):
    subconjunto = invierno.loc[invierno["zona"] == zona]
    sns.regplot(
        data=subconjunto,
        x="precipitacion_mm",
        y="superficie_agua_ha",
        ci=None,
        scatter_kws={"s": 60, "color": "#3B6FB6"},
        line_kws={"color": "#333333"},
        ax=eje,
    )
    for fila in subconjunto.itertuples():
        eje.annotate(
            str(fila.anio),
            (fila.precipitacion_mm, fila.superficie_agua_ha),
            xytext=(4, 4),
            textcoords="offset points",
            fontsize=8,
        )
    r = subconjunto["superficie_agua_ha"].corr(subconjunto["precipitacion_mm"])
    eje.set(
        title=f"{zona}\nPearson r = {r:.2f}",
        xlabel="Precipitación CHIRPS (mm)",
        ylabel="Agua abierta estimada (ha)",
    )

fig.suptitle("Precipitación invernal y agua abierta", y=1.03)
plt.tight_layout()
plt.show()

Colejuda presenta la asociación invernal más consistente. En Matanza, el coeficiente está influido por los pulsos de 2020 y 2024; en la Albufera, la precipitación no explica por sí sola la variación observada.

## Sensibilidad del umbral MNDWI

La estimación principal utiliza MNDWI mayor que cero. Los umbrales alternativos permiten evaluar cuánto cambia la superficie clasificada al incorporar respuestas espectrales mixtas o aplicar un criterio más conservador.

In [ ]:
columnas_umbral = [columna for columna in sensibilidad.columns if columna in {"-0.1", "0.0", "0.1"}]

sensibilidad_resumen = (
    sensibilidad.groupby(["zona", "temporada"])[columnas_umbral]
    .mean()
    .round(2)
)

sensibilidad_resumen

## Conclusiones

1. El análisis agregado no representa adecuadamente la diversidad interna de El Yali.
2. La Albufera mantiene agua abierta con mayor persistencia que las lagunas interiores.
3. Colejuda responde con claridad a la estacionalidad de las precipitaciones.
4. Matanza alterna pulsos de agua abierta con periodos dominados por vegetación y otras coberturas húmedas.
5. La combinación de MNDWI, NDVI, precipitación y validación visual ofrece una interpretación más sólida que el uso aislado de un único indicador.